# Digital Twin — From Foundations to a Visible Agent Loop

This foundation project combines the implementation from **Lab 3, Lab 4, and the Lab 5 Extra exercise** in the order they were taught.

The goal is to preserve the actual learning progression rather than rebuild the project from scratch:

**Digital Twin foundations → tools → first Agent Loop → Pushover tools → modularization → visible Agent Loop**

This version removes course/promotional instructions but keeps the implementation and concepts from the original lectures.


## Part 1 — Lab 3: Digital Twin Foundations

In Lab 3, I started the Digital Twin from scratch and learned how context, conversation history, tools, and an Agent Loop fit together.


### 1. Set up the building blocks

Before building the Digital Twin, I need the libraries that connect the different parts of the system.

- `dotenv` loads configuration such as API credentials from `.env`.
- `OpenAI` gives Python access to the language model.
- `PdfReader` turns my LinkedIn PDF into text that can be supplied as context.
- `gradio` will later provide the web chat interface.
- `json` becomes important when the model asks the program to execute a tool, because tool arguments arrive as JSON.

At this stage, nothing is agentic yet. I am preparing the pieces that the later system will connect together.


In [1]:
# 1. Set up the building blocks

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
from IPython.display import Markdown, display
import gradio as gr
import json


c:\Users\USER PC\projects\Agentic-AI-Engineering\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 2. Connect Python to the LLM

`load_dotenv()` reads variables stored in `.env` into the Python environment.

I then create the `OpenAI` client. My Python program is the application, while the LLM is an external service that the application communicates with through this client.

The API key is kept outside the notebook instead of being written directly into the code.


In [2]:
# 2. Connect Python to the LLM
load_dotenv(override=True)
openai = OpenAI()


### 3. Check the working directory

The notebook needs to find files such as `twin/linkedin.pdf` and `twin/summary.txt`.

Printing the current working directory and its contents helps me understand where Python is running from and whether the required files are visible. This becomes especially useful when moving the project between the course workspace and my portfolio workspace.


In [3]:
import os

print(os.getcwd())
print(os.listdir())

c:\Users\USER PC\projects\Agentic-AI-Engineering\1_foundations\Digital-twin
['digital-twin-foundations.ipynb', 'emails.txt', 'README.md', 'twin']


### 4. Turn my LinkedIn PDF into usable context

The Digital Twin needs information about the person it represents.

Here I read each page of `linkedin.pdf`, extract its text, and combine the pages into one `linkedin` string.

This is an example of **context engineering**: I deliberately provide relevant information to the model instead of expecting it to already know my professional background.


In [4]:
# 4. Turn my LinkedIn PDF into usable context
reader = PdfReader("twin/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text


### Inspect the extracted context

Printing the extracted LinkedIn text lets me verify that the PDF was read correctly before using it inside the Digital Twin.

This is a development check rather than part of the final application.


In [5]:
print(linkedin)


   
Contact
Idi-aba, Abeokuta, Ogun, Nigeria
busayo.olusanya@gmail.com
www.linkedin.com/in/solomon-
olusanya-72b191214 (LinkedIn)
www.novypro.com/profile_projects/
solomonolusanya (Portfolio)
Top Skills
Workflow Optimization
Natural Language Processing (NLP)
Microsoft Power Automate
Certifications
Data analytics 
Power Automate - Complete Guide
to Microsoft Power Automate
EF SET English Certificate 87/100
(C2 Proficient)
Career Essentials in Generative AI
by Microsoft and LinkedIn
What Is Generative AI?
Solomon Olusanya
Data Analyst || AI Data Annotator || Power BI expert || SQL ||
PYTHON || Power Automate || Agric Extensionist/Innovator
Abuja, Federal Capital Territory, Nigeria
Summary
A result-oriented Data analyst and A Graduate of Agricultural
Extension and Rural development with an innovative mindset.
A self-driven Data analyst with excellent communication and writing
ability, seeking to build a strong, 
versatile, creative and innovative career in any organization that
would supp

### 5. Add a second source of personal context

The LinkedIn profile is not the only information I want the Digital Twin to use.

`summary.txt` provides additional personal and professional context. The result is stored in `summary`, which will later be inserted into the system prompt.


In [6]:
# 5. Add a second source of personal context
with open("twin/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()


### Inspect the personal summary

This is another development check. I want to verify what was loaded before passing it to the model.


In [7]:
print(summary)


I’m Solomon.. An agriculture graduate who somehow ended up in data, AI, and now agentic AI. 😂 I’m introverted, curious, organized, and I like figuring out how things actually work. I learn best by building things, breaking them, and fixing them.
I overthink sometimes, ask a lot of “why?” questions, and prefer straightforward answers.
Fun fact: I’m also a gamer 🎮 — so if you ever need a gaming buddy, I’m probably already online. I mean its not really a fact 


### 6. Start with a basic LLM conversation

Before building an agent, I first need to understand the basic LLM interaction.

The `messages` list represents the conversation. Each message has a role:

- `system` — instructions for the model.
- `user` — the user's input.
- `assistant` — a previous model response.

This structure becomes the foundation for the Digital Twin.


In [8]:
# 6. Start with a basic LLM conversation
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi, my name is Sollie"}
]


### Send the conversation to the model

The OpenAI client sends the `messages` list to the selected model.

The API response contains more than the generated text, so `response.choices[0].message.content` extracts the actual answer.


In [9]:
response = openai.chat.completions.create(model="gpt-5.4-nano", messages=messages)
print(response.choices[0].message.content)


Hi Sollie! Nice to meet you 😊  
How can I help you today?


### 7. Change the system instructions

The system message influences how the model behaves.

Here I deliberately change the system prompt so I can see how model behavior changes when the instructions change. This is an early demonstration of why prompt design matters.


In [10]:
messages = [
    {"role": "system", "content": "You are a snarky, witty assistant"},
    {"role": "user", "content": "Hi, my name is Sollie"}
]


### Test the changed behavior

The user message is similar, but the model now receives different system instructions.

The model is therefore responding to a structured conversation rather than simply receiving an isolated question.


In [11]:
response = openai.chat.completions.create(model="gpt-5.4-nano", messages=messages)
print(response.choices[0].message.content)


Hi Sollie — nice to meet you. I’m your AI assistant. I don’t have a name tag, but I *do* have opinions about bad coffee and runaway tabs.

What can I help you with today?


### 8. Test conversation memory

I now ask a follow-up question without including the earlier statement about my name.

This demonstrates an important point: the model does not automatically remember previous requests. The application has to provide the relevant conversation history again.


In [12]:
messages = [
    {"role": "system", "content": "You are a snarky, witty assistant"},
    {"role": "user", "content": "What's my name?"}
]


### Observe the result without history

This demonstrates why conversation history has to be deliberately maintained by the application.

The model can only use information contained in the current request and the context supplied with it.


In [13]:
response = openai.chat.completions.create(model="gpt-5.4-nano", messages=messages)
print(response.choices[0].message.content)


I don’t know your name. You haven’t told me, and I can’t magically guess it out of thin air.  

Tell me what you want to be called and I’ll stick with it.


### 9. Explicitly provide conversation history

I add the earlier user message and assistant response back into `messages`.

Now the model has the information needed to answer the follow-up question using the previous exchange.

This is the basic mechanism behind conversational memory in this application: store the conversation and send the relevant history back to the model.


In [14]:
# 9. Explicitly provide conversation history
messages = [
    {"role": "system", "content": "You are a snarky, witty assistant"},
    {"role": "user", "content": "Hi, my name is Sollie"},
    {"role": "assistant", "content": "Well hi there, Sollie. It's nice to meet you."},
    {"role": "user", "content": "What's my name?"}
]


### Confirm that the model can use the supplied history

The model can now answer using information from the earlier exchange because that exchange is included in the current request.

This becomes important later when Gradio supplies conversation history to the `chat()` function.


In [15]:
response = openai.chat.completions.create(model="gpt-5.4-nano", messages=messages)
print(response.choices[0].message.content)


Your name is **Sollie**. (Yes, I’m paying attention. Allegedly.)


### 10. Build the Digital Twin's context and behavior

This is where the earlier pieces become one system.

The `system_prompt` defines the Digital Twin's role and behavior, while the personal information is inserted into the prompt.

The `f` before the string makes it a Python f-string, allowing variables such as `summary` and `linkedin` to be inserted into the larger prompt.

This is another example of **context engineering**: I am deliberately constructing the model's working context.


In [16]:
# 10. Build the Digital Twin's context and behavior
system_prompt = f"""

# Your role

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

{summary}

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a summary of the person's LinkedIn profile so that you can answer questions:

{linkedin}

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Avoid answering questions that are not related to the user's career, background, skills and experience;
steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

IMPORTANT: If you don't know the answer, say so. Never make up an answer.
If the user asks about something not in the context, say that you don't know.
"""


### Inspect the complete system prompt

Displaying the prompt lets me verify exactly what instructions and personal context the Digital Twin will receive.

This makes the relationship between my source files and the model's behavior visible.


In [17]:
display(Markdown(system_prompt))




# Your role

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

I’m Solomon.. An agriculture graduate who somehow ended up in data, AI, and now agentic AI. 😂 I’m introverted, curious, organized, and I like figuring out how things actually work. I learn best by building things, breaking them, and fixing them.
I overthink sometimes, ask a lot of “why?” questions, and prefer straightforward answers.
Fun fact: I’m also a gamer 🎮 — so if you ever need a gaming buddy, I’m probably already online. I mean its not really a fact 

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a summary of the person's LinkedIn profile so that you can answer questions:

   
Contact
Idi-aba, Abeokuta, Ogun, Nigeria
busayo.olusanya@gmail.com
www.linkedin.com/in/solomon-
olusanya-72b191214 (LinkedIn)
www.novypro.com/profile_projects/
solomonolusanya (Portfolio)
Top Skills
Workflow Optimization
Natural Language Processing (NLP)
Microsoft Power Automate
Certifications
Data analytics 
Power Automate - Complete Guide
to Microsoft Power Automate
EF SET English Certificate 87/100
(C2 Proficient)
Career Essentials in Generative AI
by Microsoft and LinkedIn
What Is Generative AI?
Solomon Olusanya
Data Analyst || AI Data Annotator || Power BI expert || SQL ||
PYTHON || Power Automate || Agric Extensionist/Innovator
Abuja, Federal Capital Territory, Nigeria
Summary
A result-oriented Data analyst and A Graduate of Agricultural
Extension and Rural development with an innovative mindset.
A self-driven Data analyst with excellent communication and writing
ability, seeking to build a strong, 
versatile, creative and innovative career in any organization that
would support maximum and efficient 
utilization of my skills geared towards the achievement of
organizational goals with skills and knowledge of Implementing and
using automated tools to extract and analyze data from primary and
secondary sources.
Use of technical tools such as: Power Bi ( data visualization)
• Python
(NumPy, Panda, Matplotlib)
• Microsoft Office tools
(Excel, Word)
• MySQL
• Power Automate
with knowledge of 
1. Removing corrupted data and fixing coding errors and related
problems.
2. Designing, developing, and implementing new ET processes
employing industry standards and best
practices to enhance loading of data from and into different sources/
target systems.
3. Developing and maintaining databases, data systems -
reorganizing data in a readable format.
Performing analysis to assess quality and meaning of data.
4. Using statistical tools to identify, analyze, and interpret patterns
and trends in complex data sets that
could be helpful for the diagnosis and prediction.
twitter: https://twitter.com/sollie_0
Email: busayo.olusanya@gmail.com
  Page 1 of 4   
Experience
Hugo
3 years 4 months
Quality Analyst
January 2025 - Present (1 year 9 months)
Lagos
Gen AI analyst 
• Efficiently utilized quality rubrics to audit and evaluate raters' annotated tasks,
ensuring alignment with project metrics.
• Conducted spot checks on annotations, providing constructive and actionable
feedback to enhance quality and accuracy.
• Identified and communicated error trends to raters, team leads, and project
managers, fostering a culture of continuous improvement.
• Facilitated coaching and training sessions to support raters in meeting quality
expectations and adhering to guidelines.
Data Annotator
June 2023 - January 2025 (1 year 8 months)
Lagos State, Nigeria
Proficiently labeled and tagged diverse data, contributing to the creation of
1000+ accurately labeled datasets for machine learning models.
• Annotated diverse data with 98.5% accuracy, delivering 50+ high-quality
datasets, and conducted thorough quality checks on 500,000+ samples,
resulting in consistent and reliable annotated datasets.
• Consistently achieved a 95% task completion rate, adeptly adapting between
tasks, and drove a 20% improvement in annotation efficiency through the
refinement of guidelines and processes.
Turing
Business Analyst Contract
January 2026 - May 2026 (5 months)
• Conducted 500+ side-by-side evaluations weekly of AI-generated outputs,
assessing factual accuracy, relevance, and response quality.
• Produced 100+ analytical justifications weekly, strengthening evaluation
consistency and RLHF feedback quality.
• Maintained 95%+ adherence to evaluation rubrics across complex fact-
checking and quality review tasks.
• Improved evaluation turnaround by 20%+ through efficient research
workflows and validation tooling.
  Page 2 of 4   
TRANSLATED.COM
Human Resources Trainer
August 2024 - January 2025 (6 months)
• Evaluated and compared responses from AI systems and human contributors
to specific prompts, ensuring adherence to task guidelines for quality.
• Provided detailed analysis of response quality based on established rating
frameworks, including Helpfulness and Accuracy.
• Contributed to refining AI training datasets through precise annotations,
improving model performance and user experience.
Devcent Trainings
Data Analyst
September 2022 - May 2023 (9 months)
Ogun, Nigeria
Educated students on SQL (Structured Query Language), Data manipulative
language (the Big 6), aggregate and windows functions, table relationship,
database design and data types.
Delivered high-quality instructions through the planning and implementation of
effective learning strategies, to increase the learning process.
TedPrime Hub
8 months
Data Analyst intern
April 2022 - August 2022 (5 months)
Abeokuta, Ogun, Nigeria
Implemented automated tools to extract and evaluate data from primary and
secondary sources such as SQL,
Incorporated Microsoft SQL, excel and power bi to evaluate a sales data set o
Data Specialist
January 2022 - August 2022 (8 months)
• Built SQL / Microsoft Power BI dashboards and analyzed datasets of 50k+
records, identifying trends that improved reporting and decision-making.
• Designed relational database models, cleaned corrupted datasets, and
automated reporting workflows using SQL, Excel, and Power BI.
• Delivered technical instruction in SQL, data modeling, aggregate/window
functions, CTEs, and database design to learners and junior analysts.
Education
  Page 3 of 4   
Federal University Of Agriculture, Abeokuta
Bachelor of Applied Science - BASc, Agricultural Business and
Management · (2014 - 2021)
Nigerian Navy secondary school Abeokuta
 · (September 2009 - June 2014)
Federal University Of Agriculture, Abeokuta
Bachelor's Degree, Development Studies
  Page 4 of 4

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Avoid answering questions that are not related to the user's career, background, skills and experience;
steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

IMPORTANT: If you don't know the answer, say so. Never make up an answer.
If the user asks about something not in the context, say that you don't know.


### 11. Give the Digital Twin a real conversation

The system prompt now defines the Digital Twin, while the user message represents the type of question a website visitor might ask.

The model now has identity, professional context, and behavioral rules rather than only a generic instruction.


In [18]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "Hi - please tell me about yourself"},
]


### Test the first Digital Twin response

The model receives the Digital Twin system prompt and generates a response using the supplied professional context.

It is still just an LLM call at this point. Tools and the Agent Loop come next.


In [19]:
response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages)
display(Markdown(response.choices[0].message.content))


Hi, I’m Solomon — well, technically this is my AI digital twin. I represent Solomon on this website and can share a bit about my background, skills, and experience.

I’m an agriculture graduate who pivoted into data analytics, AI data annotation, and now agentic AI. I’d describe myself as curious, organized, introverted, and very much the kind of person who likes to understand how things actually work. I learn best by building, testing, breaking, and fixing things.

Professionally, my background includes:
- Data analysis
- SQL
- Python
- Power BI
- Microsoft Power Automate
- NLP
- AI data annotation and quality analysis
- Workflow optimization

I’ve worked in roles across data analysis, AI annotation, quality analysis, and training, where I’ve handled tasks like:
- Cleaning and evaluating data
- Building dashboards and reports
- Reviewing AI outputs for accuracy and quality
- Improving workflows and annotation processes
- Training and supporting others on data and SQL concepts

I’m also someone who asks a lot of “why?” questions and prefers straightforward answers. And outside work, I’m a gamer too 🎮

If you’d like, I can also tell you more about my experience, technical skills, or the kind of roles I’m best suited for.

### 12. Turn the LLM call into a reusable chat function

Instead of writing the API call every time, I wrap it in `chat()`.

The function receives the new `message` and previous `history`, combines them with the system prompt, and sends the complete conversation to the model.

The function is also shaped for Gradio, which will call it whenever a user sends a message.


In [20]:
# 12. Turn the LLM call into a reusable chat function
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages)
    return response.choices[0].message.content


### Test the reusable chat function

This directly tests `chat()` without opening the web interface.

It confirms that the function can accept a message and return the Digital Twin's response.


In [21]:
chat(f"Please summarize who you are", [])


'Sure — I’m Solomon, and I’m an agriculture graduate who somehow found my way into data, AI, and now agentic AI.  \n\nI’d describe myself as:\n- **Curious and analytical**\n- **Organized and detail-oriented**\n- **Someone who likes understanding how things actually work**\n- **A builder** — I learn best by building, breaking, and fixing\n\nProfessionally, I work around **data analysis, AI data annotation, quality analysis, NLP, Power BI, SQL, Python, and Power Automate**. I’ve also spent time training others, evaluating AI outputs, and improving workflows.\n\nSo in short: I’m someone with an agriculture background who transitioned into data and AI, and I’m always looking for smarter ways to solve problems.'

### 13. Add a user interface with Gradio

Gradio provides the interface around the Python `chat()` function.

The architecture is:

**User → Gradio → `chat()` → OpenAI → response → Gradio → User**

Gradio is not the agent itself. It is the interface through which a person interacts with the Python application.


In [22]:
# 13. Add a user interface with Gradio
gr.ChatInterface(chat).launch(inbrowser=True)


* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.


### 14. Introduce the first tool

The Digital Twin now needs to do something beyond generating text.

`record_email_tool()` is a normal Python function that saves an email to `emails.txt`.

The model will eventually be allowed to **request** this Python function when the conversation requires it. The model does not execute Python directly; our application receives the request and executes the function.


In [23]:
# 14. Introduce the first tool
def record_email_tool(email):
    print(f"Tool called to record an email: {email}")
    with open("emails.txt", "a", encoding="utf-8") as f:
        f.write(email + "\n")
    return "Email received"


### Test the Python tool directly

Before involving the LLM, I test the function itself.

This separates two questions:

1. Does the Python function work?
2. Can the LLM correctly decide when and how to call it?

Testing the function first makes the later tool-calling behavior easier to understand.


In [24]:
record_email_tool("test@testy.com")


Tool called to record an email: test@testy.com


'Email received'

### 15. Describe the tool to the model

The Python function exists inside my program, but the LLM does not automatically know it exists.

The JSON schema describes the tool's name, purpose, parameter, type, and required fields.

This schema gives the model enough information to decide when it should request the tool and what argument it should provide.


In [25]:
# 15. Describe the tool to the model
record_email_tool_json = {
    "name": "record_email_tool",
    "description": "Use this tool to record that a user provided their email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The email address of this user"}
        },
        "required": ["email"],
        "additionalProperties": False
    }
}


### Make the tool available to the model

The `tools` list is the bridge between the LLM and the Python function.

By passing this structure to the API request, I tell the model that this function is available for it to request.


In [26]:
tools = [{"type": "function", "function": record_email_tool_json}]


### Inspect the tool definition

Displaying `tools` makes the bridge between the model and the Python function visible.

The model receives the schema, while the actual implementation remains inside Python.


In [27]:
tools


[{'type': 'function',
  'function': {'name': 'record_email_tool',
   'description': 'Use this tool to record that a user provided their email address',
   'parameters': {'type': 'object',
    'properties': {'email': {'type': 'string',
      'description': 'The email address of this user'}},
    'required': ['email'],
    'additionalProperties': False}}}]

### 16. Handle the first tool call

The chat function now sends the available tool along with the conversation.

If the model decides that it needs the email tool, the response indicates `tool_calls`.

The application then:

1. reads the requested tool call,
2. parses its JSON arguments,
3. executes the Python function,
4. adds the tool result back into the conversation,
5. calls the model again.

This is the first direct connection between the LLM and an external action.


In [28]:
# 16. Handle the first tool call
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
         
    if response.choices[0].finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_call = message.tool_calls[0]
            email = json.loads(tool_call.function.arguments).get("email")
            record_email_tool(email)
            messages.append(message)
            messages.append({"role": "tool", "content": "Email recorded", "tool_call_id": tool_call.id})
            response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
            
    return response.choices[0].message.content


### Test the tool-enabled Digital Twin

The Digital Twin can now be used through Gradio while also having access to the email tool.

The model decides when to request the tool; Python is responsible for actually executing it.


In [29]:
# Test the tool-enabled Digital Twin
gr.ChatInterface(chat).launch(inbrowser=True)


* Running on local URL:  http://127.0.0.1:7870
* To create a public link, set `share=True` in `launch()`.


### 17. Turn one tool call into an Agent Loop

The previous version handled one tool call and then made another model request.

But an agent may need several tool calls. The `while` loop changes the architecture to:

**LLM → tool request → Python executes tool → tool result → LLM → ...**

The cycle continues until the model stops requesting tools and produces a normal answer.

That repeated cycle is the core of the Agent Loop.


In [30]:
# 17. Turn one tool call into an Agent Loop
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
         
    while response.choices[0].finish_reason=="tool_calls":
            message = response.choices[0].message
            messages.append(message)
            for tool_call in message.tool_calls:
                email = json.loads(tool_call.function.arguments).get("email")
                record_email_tool(email)
                messages.append({"role": "tool", "content": "Email recorded", "tool_call_id": tool_call.id})
            response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
            
    return response.choices[0].message.content


### Run the first no-framework Agent

The Digital Twin now has the basic structure of an agent without using LangChain, CrewAI, or another agent framework.

Building the loop myself lets me understand the mechanics before using frameworks that abstract them away.


In [31]:
# Run the first no-framework Agent
gr.ChatInterface(chat).launch(inbrowser=True)


* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.


### Lab 3 takeaway

The important step here was not just making a chatbot. I implemented **tool calling and my first Agent Loop without an agent framework**.


## Part 2 — Lab 4: Expanding the Digital Twin

Lab 4 turns the earlier experiment into a more complete Digital Twin. The project adds Pushover notifications, two practical tools, dynamic tool handling, and a modular Python structure.


### 18. Expand the Digital Twin with practical tools

Lab 4 builds on the first Agent Loop.

This version adds Pushover notifications, a tool for recording contact details, a tool for recording unanswered questions, and dynamic tool execution.

The underlying pattern remains the same: **LLM → tool → result → LLM**.


In [32]:
# 18. Expand the Digital Twin with practical tools
# imports

from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import requests
from pypdf import PdfReader
import gradio as gr


In [33]:

load_dotenv(override=True)
openai = OpenAI()


### 19. Load Pushover configuration

The Pushover credentials are read from environment variables rather than written directly into the notebook.

The checks verify that the variables exist and look structurally correct without printing the actual secret values.


In [34]:
# 19. Load Pushover configuration

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")


Pushover user found and looks good
Pushover token found and looks good


### 20. Create a notification function

`push()` is a normal Python function that sends a message to Pushover using an HTTP request.

Later, the Digital Twin's tools can call this function when something needs to be recorded. This gives the agent an effect outside the LLM itself.


In [35]:
# 20. Create a notification function
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)


### Test the Pushover connection

This direct call checks the notification function before connecting it to the agent tools.

Testing the underlying function separately makes later agent behavior easier to debug.


In [36]:
push("HEY!!")


Push: HEY!!


### 21. Tool for recording contact interest

`record_user_details()` records when a visitor wants to get in touch.

The function sends the information to Pushover and returns a result to the application.

The important concept is that a tool can wrap an external action while presenting a simple function interface to the Agent Loop.


In [37]:
# 21. Tool for recording contact interest
def record_user_details(email, name="Name not provided", notes="not provided"):
    push(f"Recording interest from {name} with email {email} and notes {notes}")
    return "OK"


### 22. Tool for unanswered questions

The Digital Twin should not invent information when it does not know something.

This tool creates a feedback path: when the model encounters a question it cannot answer from its context, it can record that question for later review.


In [38]:
# 22. Tool for unanswered questions
def record_unknown_question(question):
    push(f"Recording {question} asked that I couldn't answer")
    return "OK"


### 23. Describe the contact tool to the LLM

The tool schema tells the model when the contact tool should be used and what information it needs.

The Python function defines **what actually happens**; the schema tells the **LLM what the function can do and what arguments it expects**.


In [39]:
# 23. Describe the contact tool to the LLM
record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record that a user is interested in being in touch and provided an email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The email address of this user"},
            "name": {"type": "string", "description": "The user's name, if they provided it"},
            "notes": {"type": "string", "description": "Any additional info about the conversation that's worth recording to give context"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}


### Describe the unknown-question tool

This schema gives the model another available action.

The description matters because the model uses the available tool descriptions when deciding whether a tool is appropriate.


In [40]:
# Describe the unknown-question tool
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {"type": "string", "description": "The question that couldn't be answered"},
        },
        "required": ["question"],
        "additionalProperties": False
    }
}


### Give the model both tools

The model now has two possible actions:

- record a visitor who wants to make contact;
- record a question the Digital Twin could not answer.

The Agent Loop decides which tool to request based on the conversation and tool descriptions.


In [41]:
tools = [{"type": "function", "function": record_user_details_json},
        {"type": "function", "function": record_unknown_question_json}]


### Inspect the complete tool set

Displaying the definitions provides a quick check that both functions have been registered correctly.


In [42]:
tools


[{'type': 'function',
  'function': {'name': 'record_user_details',
   'description': 'Use this tool to record that a user is interested in being in touch and provided an email address',
   'parameters': {'type': 'object',
    'properties': {'email': {'type': 'string',
      'description': 'The email address of this user'},
     'name': {'type': 'string',
      'description': "The user's name, if they provided it"},
     'notes': {'type': 'string',
      'description': "Any additional info about the conversation that's worth recording to give context"}},
    'required': ['email'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'record_unknown_question',
   'description': "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
   'parameters': {'type': 'object',
    'properties': {'question': {'type': 'string',
      'description': "The question that couldn't be answered"}},
    'required': ['question'],


### 24. First approach: manually map tool names to functions

The first implementation uses an explicit `if/elif` chain.

This works, but it becomes harder to maintain as the number of tools grows because every new tool requires another condition.

It is still useful as a learning step because the relationship between the model's tool name and the Python function is completely explicit.


In [43]:
# 24. First approach: manually map tool names to functions

def handle_tool_calls_with_manual_if(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)

        # THE BIG IF STATEMENT!!!

        if tool_name == "record_user_details":
            result = record_user_details(**arguments)
        elif tool_name == "record_unknown_question":
            result = record_unknown_question(**arguments)

        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results


### 25. Understand `globals()`

Python's `globals()` gives access to names defined in the current global namespace.

Here, the tool name is a string and `globals()` can look up the actual Python function with that name.

This idea allows the next version to replace a growing `if/elif` block with dynamic tool lookup.


In [44]:
# 25. Understand `globals()`
globals()["record_unknown_question"]("this is a really hard question")


Push: Recording this is a really hard question asked that I couldn't answer


'OK'

### 26. Dynamically execute the requested tool

The model's returned tool name is used to find the matching Python function.

The flow is:

**LLM chooses tool name → Python looks up function → JSON arguments become Python arguments → function executes → result goes back to LLM**

This is an important part of the no-framework Agent Loop.


In [45]:
# 26. Dynamically execute the requested tool

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else "No tool found"
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results


### 27. Reload the Digital Twin's source context

The application brings the LinkedIn profile and personal summary back into the final Digital Twin.

Keeping the context-loading step visible makes it clear what information is being supplied to the model.


In [46]:
# 27. Reload the Digital Twin's source context
reader = PdfReader("twin/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

with open("twin/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()


### 28. Refine the Digital Twin's operating rules

The final Lab 4 prompt makes the intended scope more explicit.

The Digital Twin is instructed to focus on career, background, skills, and experience. It is also told to ask for an email when someone wants to get in touch, use the contact tool, record questions it cannot answer, and avoid making up information.

This shows how **instructions + context + tools** work together to shape an agent.


In [47]:
# 28. Refine the Digital Twin's operating rules
system_prompt = f"""

# Your role

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

{summary}

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a summary of the person's LinkedIn profile so that you can answer questions:

{linkedin}

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Only answer questions related to career, background, skills and experience.
If the user asks about something unrelated, then steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

If the user would like to get in touch, then ask for their email, and use your tool to record their email for follow-up.

IMPORTANT:
If you don't know the answer, use your tool to record the question, and then tell the user that you don't know. Never make up an answer.
"""


### 29. Build the complete Lab 4 Agent Loop

The chat function sends the conversation and available tools to the model.

If the model requests tools, `handle_tool_calls()` executes them and returns the results to the conversation. The loop then calls the model again with those results.

So the complete cycle is:

**User message → LLM → tool call → Python tool → tool result → LLM → final response**


In [48]:
# 29. Build the complete Lab 4 Agent Loop
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        results = handle_tool_calls(tool_calls)
        messages.append(message)
        messages.extend(results)
        response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
    return response.choices[0].message.content


### 30. Put the complete Digital Twin behind Gradio

Gradio now sits on top of the full Lab 4 implementation.

The project has become a small application with personal context, conversation history, a user interface, tools, external notifications, dynamic tool execution, and a no-framework Agent Loop.


In [49]:
# 30. Put the complete Digital Twin behind Gradio
gr.ChatInterface(chat).launch(inbrowser=True)


* Running on local URL:  http://127.0.0.1:7872
* To create a public link, set `share=True` in `launch()`.


## Lab 4 — From Notebook to Application

The Digital Twin became more practical here: it can **record useful information, notify me through Pushover, select tools dynamically, and be split into reusable Python modules, deployment**.


The lecture moves the project toward an application that can be run outside Jupyter. This is an important transition in agent development.

The purpose of this step is to separate responsibilities so the same agent logic can be run as a real application rather than being tied to a notebook.



##  The Python Module Structure

Using the lecture we separates the Digital Twin into several Python files:

- `app.py` — application and chat interface
- `context.py` — professional/personal context
- `tools.py` — tool functions the agent can call
- `styles.py` — interface styling
- `linkedin.pdf` and `summary.txt` — the Digital Twin's source context
- `requirements.txt` — project dependencies

This separation makes the project easier to understand and maintain. Each file has a focused responsibility instead of keeping the entire application inside one notebook.



### Running the application outside the notebook

Once the code has been modularized, the Digital Twin can be started as a normal Python application.

We use:

```bash
uv run app.py
```

This is an important change in mindset. A notebook is useful while experimenting and learning, but an application needs an entry point that can be executed independently.


### Deploying the Digital Twin

We then moves from running the application locally to making it available online.

The original deployment workflow uses **Hugging Face Spaces** with Gradio. The process includes authenticating with Hugging Face, deploying the Gradio application, and configuring the application's secrets.

The deployed application needs credentials such as:

- `OPENAI_API_KEY`
- `PUSHOVER_USER`
- `PUSHOVER_TOKEN`

These should be stored as deployment secrets rather than hard-coded into the source code.

The important principle is that **deployment configuration is separate from the code itself**.


### What this production changes

Putting the Digital Twin online introduces concerns that do not appear when simply running a notebook:

- API keys must be kept private.
- Environment variables/secrets must be configured on the hosting platform.
- The application needs a reproducible dependency setup.
- The Python modules need to work together outside the notebook.
- The application needs a public interface.
- Changes to the code require redeployment.
- External services such as Pushover must also be configured correctly.

So the Digital Twin has now moved beyond an LLM experiment. It is becoming a deployable application.


### Lab 4 production takeaway

The main lesson from this part of Lab 4 is that building an agent is only one stage.

I first learned the mechanics in a notebook, then separated the application into Python modules, ran it as an application, and learned how it can be deployed with the required secrets and dependencies.

**Learning progression:**

`Notebook → Modules → Application → Deployment → Further agent improvements`


## Lab 5 Extra: A More Visible Agent Loop

This extra exercise makes the Agent Loop easier to see.

Instead of the Digital Twin's loop operating behind the chat interface, the model is given checklist tools and repeatedly uses them to plan and complete a task.


### 31. Lab 5 Extra: isolate the Agent Loop

This exercise steps away from the Digital Twin and makes the Agent Loop easier to study.

Instead of using a website-style conversation, the model gets checklist tools and must use them to plan and carry out a task.

This separates the general Agent Loop pattern from the specific Digital Twin application.


In [50]:
# 31. Lab 5 Extra: isolate the Agent Loop

from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
load_dotenv(override=True)


True

### 32. Add a display helper

`show()` gives the exercise one place to display output using Rich when available, with normal `print()` as a fallback.

This is presentation logic; it is not part of the agent's decision-making.


In [51]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)


### Create the OpenAI client

The checklist example uses the same basic pattern as the Digital Twin: Python creates an OpenAI client and uses it to communicate with the model.


In [52]:
openai = OpenAI()


### 33. Create the agent's working state

The checklist and completion lists represent state that exists outside the LLM.

The model can reason about a task, but the actual checklist state is stored and modified by Python. The tools provide controlled ways for the model to change that state.


In [53]:
# 33. Create the agent's working state

checklist = []
completed = []


### 34. Read the current checklist state

`get_checklist_report()` converts the Python state into a form that can be displayed to the user and returned to the model.

The model can therefore see which steps exist and which are completed.


In [54]:
# 34. Read the current checklist state
def get_checklist_report() -> str:
    result = ""
    for index, item in enumerate(checklist):
        if completed[index]:
            result += f"Checklist #{index + 1}: [green][strike]{item}[/strike][/green]\n"
        else:
            result += f"Checklist #{index + 1}: {item}\n"
    show(result)
    return result


### Inspect the initial checklist state

This simple test confirms that the reporting function works before the model is allowed to use it.


In [55]:
get_checklist_report()


''

### 35. Give the agent a tool for creating a plan

`create_checklist()` adds steps to the Python checklist and marks each new item as incomplete.

This is an example of an agent using a tool to create and maintain external state rather than merely producing a textual plan.


In [56]:
# 35. Give the agent a tool for creating a plan
def create_checklist(descriptions: list[str]) -> str:
    checklist.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_checklist_report()


### 36. Give the agent a tool for completing steps

`mark_complete()` changes the Python state when a checklist item is completed.

The model supplies the item index and completion notes; Python validates the index, updates the state, and returns the new checklist.


In [57]:
# 36. Give the agent a tool for completing steps
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(checklist):
        completed[index - 1] = True
    else:
        return "No checklist at this index."
    Console().print(completion_notes)
    return get_checklist_report()


### Test checklist creation directly

Before involving the LLM, I test the checklist tool itself.

This creates three items and initializes their completion state, giving me a known starting point for the next test.


In [58]:
checklist, completed = [], []

create_checklist(["Buy groceries", "Finish week 1", "Eat banana"])


Checklist #1: Buy groceries
Checklist #2: Finish week 1
Checklist #3: Eat banana

'Checklist #1: Buy groceries\nChecklist #2: Finish week 1\nChecklist #3: Eat banana\n'

### Test checklist completion directly

This marks the first checklist item as complete.

Testing the Python tool directly makes its behavior clear before the Agent Loop starts deciding when to call it.


In [59]:
mark_complete(1, "bought")


bought

Checklist #1: Buy groceries
Checklist #2: Finish week 1
Checklist #3: Eat banana

'Checklist #1: [green][strike]Buy groceries[/strike][/green]\nChecklist #2: Finish week 1\nChecklist #3: Eat banana\n'

### 37. Describe checklist creation to the model

The JSON schema tells the model how to request a new checklist.

The schema is the interface between the LLM's decision-making and the Python function that changes the checklist state.


In [60]:
# 37. Describe checklist creation to the model
create_checklist_json = {
    "name": "create_checklist",
    "description": "Add new checklist from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions of checklist items'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}


### Describe checklist completion to the model

The second schema exposes the completion action.

The model now has enough information to create a plan and mark individual steps as completed.


In [67]:
# Describe checklist completion to the model
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the checklist item at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the checklist item to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the checklist item in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}


### Register the checklist tools

Both Python functions are exposed to the LLM as available tools.

The model still cannot execute these functions itself. It can only request them; the Python Agent Loop performs the actual execution.


In [62]:
# Register the checklist tools
tools = [{"type": "function", "function": create_checklist_json},
        {"type": "function", "function": mark_complete_json}]


### 38. Reuse dynamic tool execution

This handler follows the same pattern learned in Lab 4.

The model provides a tool name and JSON arguments. `globals()` finds the corresponding Python function, the arguments are unpacked into that function, and the result is returned to the conversation.

This demonstrates that the tool-dispatch pattern is reusable beyond the Digital Twin.


In [63]:
# 38. Reuse dynamic tool execution
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results


### 39. The complete visible Agent Loop

`loop()` contains the core agent pattern in its clearest form.

It repeatedly:

1. sends the conversation and tools to the LLM,
2. checks whether the model requested tools,
3. executes those tools,
4. adds the tool results back into the messages,
5. asks the LLM what to do next.

The loop ends when the model stops requesting tools and produces its final response.


In [64]:
# 39. The complete visible Agent Loop
def loop(messages):
    response = openai.chat.completions.create(model="gpt-5.5", messages=messages, tools=tools)
    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        results = handle_tool_calls(tool_calls)
        messages.append(message)
        messages.extend(results)
        response = openai.chat.completions.create(model="gpt-5.5", messages=messages, tools=tools)
    show(response.choices[0].message.content)


### 40. Give the Agent Loop a real multi-step task

The system message tells the model to use checklist tools to plan and execute the task.

The user question is deliberately a problem that can be broken into steps. This lets me observe an agent deciding on a sequence of actions and using tools to track them before producing a final answer.


In [65]:
# 40. Give the Agent Loop a real multi-step task
system_message = """
You are given a problem to solve, by using your checklist tools to plan a list of steps, then carrying out each step in turn.
Now create a plan, set the checklist, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]


### Run the visible Agent Loop

The checklist state is reset so this run starts clean.

The model is then allowed to drive the process through the tools until it has completed its plan and can return the final solution.


In [66]:
# Run the visible Agent Loop
checklist, completed = [], []
loop(messages)


Checklist #1: Identify the missing quantity (distance between Boston and New York) and choose a reasonable 
estimate.
Checklist #2: Compute how far the Boston train travels before the New York train departs.
Checklist #3: Compute the remaining distance between the trains at 3:00 pm.
Checklist #4: Use combined speed to calculate time after 3:00 pm until they meet.
Checklist #5: Add that time to 3:00 pm and present the meeting time, noting it is based on the distance estimate.

The Boston–New York distance is not provided. I used a reasonable route-distance estimate of 215 miles.

Checklist #1: Identify the missing quantity (distance between Boston and New York) and choose a reasonable 
estimate.
Checklist #2: Compute how far the Boston train travels before the New York train departs.
Checklist #3: Compute the remaining distance between the trains at 3:00 pm.
Checklist #4: Use combined speed to calculate time after 3:00 pm until they meet.
Checklist #5: Add that time to 3:00 pm and present the meeting time, noting it is based on the distance estimate.

From 2:00 pm to 3:00 pm, the Boston train travels 60 mph × 1 h = 60 miles.

Checklist #1: Identify the missing quantity (distance between Boston and New York) and choose a reasonable 
estimate.
Checklist #2: Compute how far the Boston train travels before the New York train departs.
Checklist #3: Compute the remaining distance between the trains at 3:00 pm.
Checklist #4: Use combined speed to calculate time after 3:00 pm until they meet.
Checklist #5: Add that time to 3:00 pm and present the meeting time, noting it is based on the distance estimate.

...

Their combined closing speed is 60 + 80 = 140 mph. Time after 3:00 pm is 155 ÷ 140 = 1.1071 hours, about 1 hour 6 
minutes 26 seconds.

Checklist #1: Identify the missing quantity (distance between Boston and New York) and choose a reasonable 
estimate.
Checklist #2: Compute how far the Boston train travels before the New York train departs.
Checklist #3: Compute the remaining distance between the trains at 3:00 pm.
Checklist #4: Use combined speed to calculate time after 3:00 pm until they meet.
Checklist #5: Add that time to 3:00 pm and present the meeting time, noting it is based on the distance estimate.

Adding 1 h 6 min 26 s to 3:00 pm gives approximately 4:06 pm.

Checklist #1: Identify the missing quantity (distance between Boston and New York) and choose a reasonable 
estimate.
Checklist #2: Compute how far the Boston train travels before the New York train departs.
Checklist #3: Compute the remaining distance between the trains at 3:00 pm.
Checklist #4: Use combined speed to calculate time after 3:00 pm until they meet.
Checklist #5: Add that time to 3:00 pm and present the meeting time, noting it is based on the distance estimate.

Answer: They meet at approximately 4:06 pm.

This assumes the Boston–New York distance is about 215 miles.

Calculation:  
• Boston train travels from 2:00 to 3:00 pm: 60 miles  
• Remaining distance at 3:00 pm: 215 − 60 = 155 miles  
• Combined speed: 60 + 80 = 140 mph  
• Time to meet after 3:00 pm: 155 ÷ 140 ≈ 1.107 hours ≈ 1 hour 6 minutes  

So they meet around 4:06 pm.

## Final Learning Takeaway

This foundation project shows the progression I followed:

1. Build a Digital Twin from real personal context.
2. Give the model conversation history and a structured system prompt.
3. Connect Python functions as tools.
4. Build an Agent Loop without a framework.
5. Add practical tools and Pushover notifications.
6. Move the experiment into Python modules.
7. Make the Agent Loop visible with a separate checklist example.

**Core focus:** understanding how agentic systems work from the ground up before relying on an agent framework.
